In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: clean and split, then for each fold run through the embedding and save the vectors.
# False: skip that work. The tuning loop loads those saved fold frames.
REBUILD = False

spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","4g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 12:03:36 WARN Utils: Your hostname, NineQueendeMacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.11.225.55 instead (on interface en0)
26/11/11 12:03:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 12:03:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/11 12:03:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
ANALYTICS_MODEL = "improved_baseline"
EMBEDDING_MODEL = "baseline"
K = 5
# Holdout score to maximize: "weighted_f1" or "accuracy".
METRIC = "weighted_f1"

PARA_RANGE = {
    "regParam": [0.3, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0, 50.0, 100.0],
}

In [3]:
from src.cleaning import data_cleaning
from src.embeddings import EmbeddingModel
from src.encoder import encoder
from src.k_flod import KFold
from src.split import split, weight
from src.text_features import text_feature_engineering, fit_phrases


embedding_train = getattr(EmbeddingModel, f"{EMBEDDING_MODEL}_train")
embedding_predict = getattr(EmbeddingModel, f"{EMBEDDING_MODEL}_predict")

if REBUILD:
    with data_oi.scope():
        df_raw = data_oi.load_raw_data()
        df_cleaned = data_cleaning(df_raw)
        df_cleaned.cache().count()
        df_engineered = text_feature_engineering(df_cleaned, True)
        df_engineered.cache().count()
        df_train, df_valid, df_test = split(df_engineered, SEED)
        fold_ids = iter(range(K))

        def prepare_fold(fit, holdout):
            fold = next(fold_ids)
            print(f"Fold {fold}")
            weighted = weight(fit)
            encoded_fit, encoded_holdout = encoder(weighted, holdout)
            encoded_fit.count()
            encoded_holdout.count()
            data_oi.unpersist(weighted)
            phrases = fit_phrases(encoded_fit)
            embedding_model = EmbeddingModel(
                encoded_fit, encoded_holdout, encoded_fit.limit(1), phrases
            )
            model_name = f"{EMBEDDING_MODEL}_para_{fold}"
            print(f"Fold {fold} Embedding Model `{EMBEDDING_MODEL}` Training")
            embedding_train(embedding_model, model_name)
            print(f"Fold {fold} Embedding Model `{EMBEDDING_MODEL}` Running")
            train_embedding, valid_embedding, test_embedding = embedding_predict(
                embedding_model, model_name
            )
            data_oi.unpersist(encoded_fit, encoded_holdout, test_embedding)
            for extra in (
                embedding_model._train_expanded,
                embedding_model._train_expanded_phrase,
            ):
                if extra is not None:
                    data_oi.unpersist(extra)
            del embedding_model, phrases
            return train_embedding, valid_embedding

        folds = KFold(f"{EMBEDDING_MODEL}_para", df_train, K, prepare_fold, SEED + 1)
else:
    folds = KFold.open(f"{EMBEDDING_MODEL}_para", K)

In [4]:
from itertools import product

from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import functions as F

from src.analytics import AnalyticsModel

if METRIC not in {"weighted_f1", "accuracy"}:
    raise ValueError("METRIC must be 'weighted_f1' or 'accuracy'")

analytics_function = getattr(AnalyticsModel, ANALYTICS_MODEL)
range_keys = list(PARA_RANGE)
candidates = [
    dict(zip(range_keys, values))
    for values in product(*(PARA_RANGE[key] for key in range_keys))
]
evaluator = MulticlassClassificationEvaluator(
    labelCol="rating",
    predictionCol="rating_pred",
    metricName="accuracy" if METRIC == "accuracy" else "f1",
)

def holdout_score(pred) -> float:
    scored = pred.withColumn("rating_pred", F.col("prediction") + 1)
    return evaluator.evaluate(scored)

In [5]:
best_para = None
best_score = None

for trial in candidates:
    folds.reset()
    fold_scores = []

    for i in range(K):
        with data_oi.scope():
            fit_embedding, holdout_embedding = folds.next_fold()
            model = AnalyticsModel(
                EMBEDDING_MODEL, fit_embedding, holdout_embedding, fit_embedding.limit(1)
            )
            _, holdout_pred, _ = analytics_function(model, trial)
            fold_scores.append(holdout_score(holdout_pred))
            model.release_features()
            del fit_embedding, holdout_embedding, holdout_pred, model

    score = sum(fold_scores) / K
    print(trial, score)
    if best_score is None or score > best_score:
        best_score = score
        best_para = trial
        data_oi.save_analytics_para(EMBEDDING_MODEL, ANALYTICS_MODEL, best_para)
        print(f"saved best {METRIC}: {best_score}")

para = data_oi.load_analytics_para(EMBEDDING_MODEL, ANALYTICS_MODEL)
print(f"best {METRIC}: {best_score}")
print(para)

{'regParam': 0.3} 0.37685257389265275
saved best weighted_f1: 0.37685257389265275


{'regParam': 0.5} 0.3795820947884772
saved best weighted_f1: 0.3795820947884772


{'regParam': 1.0} 0.37990666578297255
saved best weighted_f1: 0.37990666578297255


{'regParam': 1.5} 0.378479290974997


{'regParam': 2.0} 0.3764230085481999


{'regParam': 3.0} 0.37354171270739744


... (4 outputs left out)

{'regParam': 20.0} 0.3629121238012914


{'regParam': 30.0} 0.3620121811619089


{'regParam': 50.0} 0.3611015377422614


{'regParam': 100.0} 0.36007055971527924
best weighted_f1: 0.37990666578297255
{'regParam': 1.0}
